<a href="https://colab.research.google.com/github/debashisdotchatterjee/BPAI_viewpoint1/blob/main/BPAI_Colab_simulation_%2Breal_data_FULL_CODE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Population-Aware AI: corrected Colab research pipeline (v2)

**Complete, self-contained notebook.** The second code cell writes the entire research source to `bpai_study.py`; no companion upload is needed.

**Scientific design:** survey-weighted 2017 Alabama `ACSEmployment` classifier; full-person 2018 Alabama, California and Texas survey audits, official 80 ACS replicate weights, weighted population TPR gaps, hierarchical weighted pseudo-posterior Laplace approximations, mathematical verification and Monte Carlo sampling simulations.

**Critical change:** the 2018 ACS files are downloaded with response/ZIP/CSV validation and read in 50,000-row chunks; the prior downloader could silently suppress network failures. This notebook reports full exceptions and packages diagnostics even if a stage fails. **ACS results are real only if the official Census download succeeds.**

**How to use:** Run cells 1–5 in order. `RUN_PROFILE="pilot"` checks the pipeline; after success, change it to `"full"` for publication-grade Monte Carlo size and rerun the execution cell (the dataset cache is reused). `"acs_only"` and `"simulation_only"` are also available. The resulting ZIP is automatically offered for download at the end.

In [1]:
# In Colab this installs packages in the active notebook Python environment.
%pip -q install "numpy>=1.24" "pandas>=2.0" "scipy>=1.11" "scikit-learn>=1.3" "matplotlib>=3.7" "folktables>=0.0.12" "requests>=2.28" "nbformat>=5.7"

## Source: complete analytical implementation

This code cell **must run before** the execution cell. It writes a standalone Python script containing the mathematical tests, simulations, robust Census downloader, chunk-wise ACS design replication, Bayesian approximations, figures, tables, and manifest.

In [2]:
%%writefile bpai_study.py
#!/usr/bin/env python3
"""BPAI: reproducible simulations + ACS 2018 population-aware fairness audits.

Methods align with the manuscript 'Toward Bayesian Population-Aware AI'.

Requires: numpy, pandas, scipy, scikit-learn, matplotlib, folktables (ACS mode).
See README.md for design/epistemic cautions. Python >= 3.10.
"""
from __future__ import annotations
import argparse
import gc
import hashlib
import json
import math
import os
import platform
import random
import sys
import warnings
import zipfile
import traceback
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize
from scipy.special import expit, logit
from scipy.stats import norm
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

G_NAMES = ('White alone (RAC1P=1)', 'Black alone (RAC1P=2)')
C_NAMES = ('16-34 / male', '16-34 / female', '35-54 / male',
           '35-54 / female', '55+ / male', '55+ / female')
NC = len(C_NAMES)
COLORS = {'naive':'#9b5a1a', 'design':'#194e7a', 'pseudo':'#777ba2',
          'hier':'#347c74', 'adjusted':'#a33753',
          'AL':'#194e7a', 'CA':'#347c74', 'TX':'#a33753'}
METHODS = ('naive', 'design', 'pseudo', 'hier', 'adjusted')


def style():
    plt.rcParams.update({
        'font.family':'DejaVu Sans','font.size':10,'axes.labelsize':10.5,
        'axes.titlesize':11.5,'axes.titleweight':'semibold',
        'axes.spines.right':False,'axes.spines.top':False,
        'axes.edgecolor':'#46505b','axes.labelcolor':'#243746',
        'xtick.color':'#334452','ytick.color':'#334452',
        'figure.facecolor':'white','axes.facecolor':'white',
        'savefig.facecolor':'white','pdf.fonttype':42,'ps.fonttype':42,
        'lines.linewidth':1.9, 'grid.alpha':0.20,
    })


def export_figure(fig, directory, stem):
    p = Path(directory); p.mkdir(parents=True,exist_ok=True)
    for ext in ('png', 'pdf', 'svg'):
        fig.savefig(p / (stem+'.'+ext), dpi=300, bbox_inches='tight', pad_inches=0.13)
    plt.close(fig)


def table(df: pd.DataFrame, directory, name: str):
    p=Path(directory); p.mkdir(parents=True,exist_ok=True)
    df.to_csv(p/(name+'.csv'),index=False,float_format='%.9g')
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            (p/(name+'.tex')).write_text(df.to_latex(index=False,escape=True,
                    float_format=lambda z: f'{z:.4f}'),encoding='utf-8')
    except Exception as ex:
        warnings.warn(f'Could not export latex table {name}: {ex}')


def safe_ratio(num,den):
    a=np.asarray(num,dtype=float); b=np.asarray(den,dtype=float)
    result=np.divide(a,b,out=np.full(np.broadcast_shapes(a.shape,b.shape),np.nan),where=b>0)
    return result.item() if result.ndim==0 else result


def weighted_gap(y,z,g,w,group_codes=(1,2)):
    vals=[]; dens=[]
    for code in group_codes:
        k=(np.asarray(y)==1)&(np.asarray(g)==code)
        dw=float(np.sum(np.asarray(w)[k])); dens.append(dw)
        vals.append(float(safe_ratio(np.sum(np.asarray(w)[k]*np.asarray(z)[k]),dw)))
    return float(vals[0]-vals[1]), np.array(vals), np.array(dens)


def sdr_variance(theta0, replicates):
    """Official ACS PUMS SDR factor: (4/80) sum (theta_r-theta0)^2."""
    rep=np.asarray(replicates,dtype=float)
    if rep.shape != (80,) or not np.all(np.isfinite(rep)) or not np.isfinite(theta0):
        return np.nan
    return float((4.0/80)*np.sum((rep-theta0)**2))


def replicate_gap(y,z,g,repweights):
    """Entire weighted numerator and denominator recomputed for all 80 columns."""
    w=np.asarray(repweights,dtype=float)
    assert w.ndim==2 and w.shape[1]==80
    rates=[]
    for code in (1,2):
        subset=(y==1)&(g==code)
        sw=w[subset]
        den=sw.sum(axis=0)
        num=(sw*np.asarray(z)[subset,None]).sum(axis=0)
        rates.append(safe_ratio(num,den))
    return rates[0]-rates[1], np.stack(rates,axis=0)


def poisson_linearized_gap_variance(y,z,g,pi,draw,w):
    """Poisson-sampling HT-ratio linearization for a fixed finite-population h.
    pi is true independent inclusion propensity. Conditional on original pop.
    """
    variances=[]
    for code in (1,2):
        k=draw & (y==1) & (g==code)
        dk=float(w[k].sum())
        if dk<=0: return np.nan
        tk=float(np.sum(w[k]*z[k])/dk)
        variances.append(np.sum((1-pi[k])*(z[k]-tk)**2/(pi[k]**2))/dk**2)
    return float(sum(variances))


def collect_cells(y,z,g,c,w, nc=NC):
    """Weighted cell positive successes and denominators + raw counts.

    All Y-positive cells define the denominator of TPR, not all group members.
    Weights for posterior pseudo-counts normalized to mean weight one globally.
    """
    y=np.asarray(y); z=np.asarray(z); g=np.asarray(g); c=np.asarray(c)
    w=np.asarray(w,dtype=float)
    if not (len(y)==len(z)==len(g)==len(c)==len(w)):
        raise ValueError('Misaligned arrays')
    if not np.all(np.isfinite(w)) or not np.all(w>0):
        raise ValueError('Expected strictly positive finite audit weights')
    if not np.all(np.isin(z,[0,1])) or not np.all(np.isin(y,[0,1])):
        raise ValueError('Y and Z must be binary')
    if not np.all((c>=0)&(c<nc)):
        raise ValueError('Invalid cell code')
    group=np.asarray(g,dtype=int)
    k=(y==1)&np.isin(group,[1,2]); ix=(group[k]-1)*nc+c[k]
    den=np.bincount(ix,weights=w[k],minlength=2*nc).reshape(2,nc)
    yes=np.bincount(ix,weights=w[k]*z[k],minlength=2*nc).reshape(2,nc)
    n=np.bincount(ix,minlength=2*nc).reshape(2,nc)
    if not np.all(yes<=den+1e-6): raise ValueError('Success > count')
    return den,yes,n


def cell_design(nc=NC):
    """Eta_gc = intercept + group intercept + cell + group:cell interaction.
    Priors keep this overparameterized random-effects basis identifiable.
    """
    npar=1+2+nc+2*nc
    x=np.zeros((2*nc,npar))
    for g in range(2):
        for c in range(nc):
            row=g*nc+c
            x[row,0]=1
            x[row,1+g]=1
            x[row,3+c]=1
            x[row,3+nc+row]=1
    return x


def hierarchical_laplace(den,yes,draws,rng,prior_scale=0.8):
    """Gaussian-prior hierarchical-logit weighted pseudo posterior, Laplace.
    Fixed prior scales, not hyperparameter-learning full Bayes. All weighted
    likelihood contributions are normalized by the original sample mean weight.
    The returned approximation is not automatically design calibrated.
    """
    den=np.asarray(den,float); yes=np.asarray(yes,float)
    nc=den.shape[1]; X=cell_design(nc)
    xden=den.ravel(); xy=yes.ravel()
    assert xden.shape==(2*nc,) and np.all(xy>=-1e-7)
    s=np.r_[2.5,np.full(2,prior_scale),np.full(nc,prior_scale),
                  np.full(2*nc,0.75*prior_scale)]
    prec=1.0/(s*s)
    pooled=float(np.sum(xy)/max(np.sum(xden),1))
    start=np.zeros(X.shape[1]); start[0]=float(logit(np.clip(pooled,.001,.999)))
    def val_grad(b):
        eta=X@b
        p=expit(eta)
        f=float(np.sum(xden*np.logaddexp(0,eta)-xy*eta)+0.5*np.sum(prec*b*b))
        grad=X.T@(xden*p-xy)+prec*b
        return f,grad
    opt=minimize(val_grad,start,method='L-BFGS-B',jac=True,
                 options={'maxiter':500,'ftol':1e-12,'gtol':1e-7})
    if not np.all(np.isfinite(opt.x)):
        raise RuntimeError('Laplace MAP optimizer produced nonfinite parameters')
    bhat=opt.x; p=expit(X@bhat)
    H=(X.T*(xden*p*(1-p)))@X+np.diag(prec)
    cf=cho_factor(H,lower=True,check_finite=True)
    cov=cho_solve(cf,np.eye(len(bhat)))
    # Symmetrization removes numerical floating point asymmetry
    cov=(cov+cov.T)/2
    jitter=1e-11
    L=np.linalg.cholesky(cov+np.eye(len(bhat))*jitter)
    beta=bhat[None,:]+rng.standard_normal((draws,len(bhat)))@L.T
    theta=expit(beta@X.T).reshape(draws,2,nc)
    return theta,{'converged':bool(opt.success),'objective':float(opt.fun),
                  'prior_scale':float(prior_scale)}


def pseudo_draws(den,yes,n_draws,rng):
    """Independent Beta pseudo-posteriors for cell rates; not hierarchical."""
    return rng.beta(1+yes[None,:,:],1+(den-yes)[None,:,:],
                    size=(n_draws,)+den.shape)


def composition_draws(den,ndraw,rng):
    """Dirichlet pseudo-posterior for positive-outcome cell compositions.
    The source of the target positive outcome composition must be specified.
    """
    return np.stack([rng.dirichlet(1+den[g],size=ndraw) for g in range(2)],axis=1)


def posterior_gap(theta,q):
    return np.sum(theta[:,0]*q[:,0],axis=1)-np.sum(theta[:,1]*q[:,1],axis=1)


def design_adjust_gap(draws,anchor,var_design):
    """Scalar variance alignment; NOT a coherent design-correct Bayesian posterior."""
    draws=np.asarray(draws,float)
    v=float(np.var(draws,ddof=1))
    if not (np.isfinite(var_design) and var_design>=0 and v>0):
        return np.full(len(draws),np.nan)
    # Normalized-to-design covariance for the scalar estimand only.
    return np.clip(anchor+(draws-draws.mean())*np.sqrt(var_design/v),-1,1)


def summarize_draws(draws,eps=0.05):
    draws=np.asarray(draws,float); draws=draws[np.isfinite(draws)]
    if not len(draws): return {'mean':np.nan,'lo':np.nan,'hi':np.nan,'sd':np.nan,'risk':np.nan}
    a,b=np.quantile(draws,[.025,.975])
    return {'mean':float(np.mean(draws)),'lo':float(a),'hi':float(b),
            'sd':float(np.std(draws,ddof=1)),
            'risk':float(np.mean(np.abs(draws)>eps))}


def certificate(q,theta,alpha=0.05,eps=0.05):
    """Construct a joint credible event via 1-alpha/2 max-statistic bounds.
    By the union bound, joint coverage is >=1-alpha (up to Monte Carlo error).
    q,theta dimensions (draw,group,cell).
    """
    refq=q.mean(axis=0); reft=theta.mean(axis=0)
    refgap=float(np.sum(refq[0]*reft[0])-np.sum(refq[1]*reft[1]))
    tv=0.5*np.abs(q-refq[None,:,:]).sum(axis=2)
    tdev=np.abs(theta-reft[None,:,:])
    # max over groups and group-cells for truly simultaneous event
    eta=float(np.quantile(np.max(tv,axis=1),1-alpha/2))
    erate=float(np.quantile(np.max(tdev,axis=(1,2)),1-alpha/2))
    B=float(2*eta+2*erate)  # each group qbar sums to one
    empirical_joint=np.mean((np.max(tv,axis=1)<=eta)&
                             (np.max(tdev,axis=(1,2))<=erate))
    if abs(refgap)+B<=eps: label='certified_low_risk'
    elif abs(refgap)-B>eps: label='certified_high_risk'
    else: label='inconclusive'
    # Numerical check: theorem must hold drawwise on E.
    delta=posterior_gap(theta,q)
    on_event=(np.max(tv,axis=1)<=eta)&(np.max(tdev,axis=(1,2))<=erate)
    if np.any(np.abs(delta[on_event]-refgap)>B+1e-9):
        raise AssertionError('Certificate inequality violated')
    return {'refgap':refgap,'B':B,'eta':eta,'e':erate,'joint_prob':float(empirical_joint),
            'status':label,'alpha':alpha,'epsilon':eps}


def decomposition(qP,tP,qQ,tQ):
    """Exact descriptive composition and conditional-rate decomposition."""
    qP=np.asarray(qP); tP=np.asarray(tP); qQ=np.asarray(qQ); tQ=np.asarray(tQ)
    comp=np.sum((qQ-qP)*tP,axis=1)
    rate=np.sum(qQ*(tQ-tP),axis=1)
    tprP=np.sum(qP*tP,axis=1); tprQ=np.sum(qQ*tQ,axis=1)
    dif=tprQ-tprP
    if not np.allclose(dif,comp+rate,atol=1e-10):
        raise AssertionError('Incorrect composition-rate decomposition')
    max_change=sum([.5*abs(qQ[i]-qP[i]).sum()+
                    np.sum(qQ[i]*np.abs(tQ[i]-tP[i])) for i in range(2)])
    if abs(float(dif[0]-dif[1]))>max_change+1e-10:
        raise AssertionError('Transport total variation bound violated')
    dg= dif[0]-dif[1]
    return {'composition_A':float(comp[0]),'composition_B':float(comp[1]),
            'rate_A':float(rate[0]),'rate_B':float(rate[1]),
            'composition_gap':float(comp[0]-comp[1]),
            'rate_gap':float(rate[0]-rate[1]),
            'delta_source':float(tprP[0]-tprP[1]),
            'delta_target':float(tprQ[0]-tprQ[1]),'change':float(dg),
            'tv_bound':float(sum([.5*abs(qQ[i]-qP[i]).sum()+
                    np.sum(qQ[i]*np.abs(tQ[i]-tP[i])) for i in range(2)]))}


def ignorance_bounds(q,theta,supported):
    """Sharp interval: known target q, oracle identified supported-cell rates.
    Unsupported rates free in [0,1], NOT filled by the hierarchical model.
    """
    known=np.sum(q*theta*supported,axis=1)
    miss=np.sum(q*(~supported),axis=1)
    lo=float(known[0]-known[1]-miss[1]); hi=float(known[0]+miss[0]-known[1])
    return lo,hi,float(sum(miss))


def fixed_synthetic_population(N,rng):
    """Construct finite-population ground truth with known fixed predictions."""
    g=rng.choice([1,2],size=N,p=[0.62,0.38]); c=np.empty(N,dtype=np.int16)
    qc=np.array([[.08,.12,.24,.25,.19,.12],[.19,.16,.22,.16,.16,.11]])
    for j in (1,2):
        k=(g==j);c[k]=rng.choice(NC,size=k.sum(),p=qc[j-1])
    prev=np.array([[.23,.33,.62,.66,.48,.43],
                   [.19,.24,.51,.54,.34,.30]])
    rate=np.array([[.74,.80,.86,.90,.81,.79],
                   [.69,.75,.81,.85,.74,.73]])
    y=(rng.random(N)<prev[g-1,c]).astype(np.int8)
    fp=.12
    z=(rng.random(N)<np.where(y==1,rate[g-1,c],fp)).astype(np.int8)
    truth,_,_=weighted_gap(y,z,g,np.ones(N))
    return {'g':g,'c':c,'y':y,'z':z,'truth':truth,'N':N}


def incl_propensity(pop,scenario):
    g=pop['g'];c=pop['c'];y=pop['y']
    if scenario=='near_uniform': p=np.full(len(g),.12)
    elif scenario=='informative':
        p=.13*np.where(g==1,1.4,.68)*np.array([1.6,.8,1.45,.65,1.1,.55])[c]*np.where(y==1,1.4,.75)
    elif scenario=='strong_informative':
        p=.13*np.where(g==1,2.5,.4)*np.array([2.4,.38,2,.4,1.3,.30])[c]*np.where(y==1,1.9,.65)
    elif scenario=='prediction_dependent':
        p=.13*np.where(g==1,1.6,.65)*np.array([1.6,.8,1.45,.65,1.1,.55])[c]*np.where(y==1,1.4,.75)*np.where(pop['z']==1,1.8,.56)
    elif scenario=='lack_support':
        p=.13*np.where(g==1,1.4,.68)*np.array([1.6,.8,1.45,.65,1.1,.55])[c]*np.where(y==1,1.4,.75)
        p[(g==2)&(c==5)]=0.0
    else: raise ValueError(scenario)
    return np.where(p==0,0,np.clip(p,.015,.72))


def simulate_once(pop,pi,rng,ndraw,eps,fit_hier=True):
    y=pop['y'];z=pop['z'];g=pop['g'];c=pop['c']
    sel=rng.random(len(y))<pi
    if sel.sum()<100: return []
    w=1/pi[sel]; yy=y[sel];zz=z[sel];gg=g[sel];cc=c[sel]
    unwt,unrates,_=weighted_gap(yy,zz,gg,np.ones(len(w)))
    npos=np.array([int(np.sum((yy==1)&(gg==code))) for code in (1,2)])
    naive_se=float(np.sqrt(np.sum(unrates*(1-unrates)/np.maximum(npos,1))))
    wt,_,_=weighted_gap(yy,zz,gg,w)
    if not (np.isfinite(unwt) and np.isfinite(wt)): return []
    variance=poisson_linearized_gap_variance(y,z,g,pi,sel,1/np.maximum(pi,1e-20))
    se=math.sqrt(variance) if np.isfinite(variance) and variance>=0 else np.nan
    scale=float(np.mean(w)); den,yes,cnt=collect_cells(yy,zz,gg,cc,w/scale)
    # Under no support, all cell-based posterior approaches rely on extrapolation.
    has_support=bool((cnt>=1).all())
    q=composition_draws(den,ndraw,rng)
    pt=pseudo_draws(den,yes,ndraw,rng)
    pdra=posterior_gap(pt,q)
    if fit_hier:
        ht,_=hierarchical_laplace(den,yes,ndraw,rng)
        hdra=posterior_gap(ht,q)
    else: hdra=np.full(ndraw,np.nan)
    adj=design_adjust_gap(hdra,wt,variance)
    n=len(w); true=pop['truth']
    out=[]
    rows={
        'naive':{'mean':unwt,'lo':unwt-1.96*naive_se,'hi':unwt+1.96*naive_se,
                 'risk':np.nan},
        'design':{'mean':wt,'lo':wt-1.96*se,'hi':wt+1.96*se,
                  'risk':float(1-norm.cdf((eps-wt)/se)+norm.cdf((-eps-wt)/se)) if se>0 else np.nan},
        'pseudo':summarize_draws(pdra,eps),
        'hier':summarize_draws(hdra,eps),
        'adjusted':summarize_draws(adj,eps),
    }
    for name,row in rows.items():
        out.append({'scenario':'','method':name,'n':n,'true':true,'est':row['mean'],
                    'bias':row['mean']-true,'covered':int(row['lo']<=true<=row['hi']) if np.isfinite(row['lo']) else np.nan,
                    'width':row['hi']-row['lo'],'rho':row['risk'],
                    'false_low_risk':int(row['risk']<=.10 and abs(true)>eps) if np.isfinite(row['risk']) else np.nan,
                    'false_high_risk':int(row['risk']>=.90 and abs(true)<=eps) if np.isfinite(row['risk']) else np.nan,
                    'support_complete':has_support})
    return out


def synthetic_reversal(figdir, tabdir):
    th=np.array([[.90,.50],[.85,.55]])
    qP=np.array([[.20,.80],[.80,.20]]);qQ=np.array([[.70,.30],[.30,.70]])
    d=decomposition(qP,th,qQ,th)
    table(pd.DataFrame([d]),tabdir,'toy_transport_decomposition')
    fig,ax=plt.subplots(figsize=(7.5,4.2))
    for i,(lbl,v) in enumerate([('Observed sample',d['delta_source']),
                                 ('Target population',d['delta_target'])]):
        ax.barh(i,v,color=COLORS['naive'] if i==0 else COLORS['design'],height=.42)
        ax.text(v+(.012 if v>0 else -.012),i,f'{v:+.2f}',
                ha='left' if v>0 else 'right',va='center',fontweight='bold')
    ax.axvline(0,color='#333',lw=.8);ax.set_yticks([0,1],['Observed sample','Target population'])
    ax.invert_yaxis();ax.set_xlim(-.29,.23);ax.set_xlabel('White minus Black true-positive rate')
    ax.set_title('An exact sign reversal from population composition alone')
    export_figure(fig,figdir,'S01_hypothetical_sign_reversal')
    return d


def simulate(out,reps=120,N=24000,draws=750,seed=2088,eps=.05,scenarios=None):
    path=Path(out); figs=path/'figures';tabs=path/'tables'
    figs.mkdir(parents=True,exist_ok=True);tabs.mkdir(parents=True,exist_ok=True)
    rg=np.random.default_rng(seed)
    synthetic_reversal(figs,tabs)
    scenarios=scenarios or ['near_uniform','informative','strong_informative','prediction_dependent','lack_support']
    allrows=[]; supportrows=[]
    for s in scenarios:
        pop=fixed_synthetic_population(N,rg)
        pi=incl_propensity(pop,s)
        trueq,trueyes,_=collect_cells(pop['y'],pop['z'],pop['g'],pop['c'],np.ones(N))
        q=safe_ratio(trueq,trueq.sum(axis=1,keepdims=True))
        theta=safe_ratio(trueyes,trueq)
        supported=np.ones((2,NC),dtype=bool)
        for group in (1,2):
            for cell in range(NC):
                sub=(pop['g']==group)&(pop['c']==cell)&(pop['y']==1)
                supported[group-1,cell]=bool(np.any(pi[sub]>0))
        lo,hi,missing=ignorance_bounds(q,np.nan_to_num(theta,nan=0),supported)
        supportrows.append({'scenario':s,'truth':pop['truth'],
                            'sharp_lower':lo,'sharp_upper':hi,
                            'unknown_positive_mass':missing,
                            'n_population':N,'expected_sample_n':float(pi.sum())})
        for r in range(reps):
            vals=simulate_once(pop,pi,rg,draws,eps)
            for row in vals: row.update(scenario=s,replication=r+1)
            allrows.extend(vals)
        # Persist a recoverable checkpoint after each completed scenario.
        checkpoint=path/'checkpoints';checkpoint.mkdir(parents=True,exist_ok=True)
        table(pd.DataFrame([r for r in allrows if r['scenario']==s]),checkpoint,f'{s}_replications')
        table(pd.DataFrame([supportrows[-1]]),checkpoint,f'{s}_identification')
        print(f'[simulation] {s}: {reps} draws, population gap = {pop["truth"]:+.4f}',flush=True)
    detail=pd.DataFrame(allrows)
    if detail.empty: raise RuntimeError('Simulation produced no valid replicates')
    summary=detail.groupby(['scenario','method'],as_index=False).agg(
        n_runs=('bias','count'),mean_est=('est','mean'),bias=('bias','mean'),
        mcse_bias=('bias',lambda z:float(z.std(ddof=1)/np.sqrt(len(z)))) ,
        mean_n=('n','mean'),
        rmse=('bias',lambda z:float(np.sqrt(np.mean(z**2)))),
        coverage=('covered','mean'),
        coverage_mcse=('covered',lambda z:float(np.sqrt(z.mean()*(1-z.mean())/len(z)))),
        mean_interval_width=('width','mean'),
        false_low_risk_rate=('false_low_risk','mean'),
        false_high_risk_rate=('false_high_risk','mean'),
        proportion_complete_cells=('support_complete','mean'))
    table(detail,tabs,'simulation_replication_level')
    table(summary,tabs,'simulation_method_summary')
    table(pd.DataFrame(supportrows),tabs,'simulation_partial_identification')
    # Figure: RMSE by scenario
    sc=[s for s in scenarios if s!='lack_support']
    fig,ax=plt.subplots(figsize=(8.3,4.5))
    x=np.arange(len(sc));width=.15
    for k,m in enumerate(METHODS):
        vals=[float(summary.query('scenario == @s and method == @m').rmse.iloc[0]) for s in sc]
        ax.bar(x+(k-2)*width,vals,width=width*.91,color=COLORS[m],label=m)
    ax.set_xticks(x, [s.replace('_',' ').title() for s in sc]); ax.set_ylabel('RMSE of estimated population gap')
    ax.set_title('Design-based Monte Carlo estimation accuracy')
    ax.legend(ncol=5,loc='upper center',bbox_to_anchor=(.5,-.12),frameon=False,fontsize=9)
    ax.grid(axis='y');export_figure(fig,figs,'S02_simulation_RMSE')
    # Figure: empirical 95% coverage and target
    fig,ax=plt.subplots(figsize=(8.3,4.5))
    for k,m in enumerate(METHODS):
        vals=[float(summary.query('scenario == @s and method == @m').coverage.iloc[0]) for s in sc]
        ax.plot(sc,vals,marker=['o','s','^','D','v'][k],color=COLORS[m],label=m,ms=6)
    ax.axhline(.95,ls='--',color='#333',lw=1,label='Nominal 0.95')
    ax.set_ylim(0,1.035);ax.set_ylabel('Repeated-Poisson-design coverage')
    ax.set_xticks(range(len(sc)),[s.replace('_',' ').title() for s in sc]);ax.grid(axis='y')
    ax.set_title('Do nominal intervals cover the fixed finite-population gap?')
    ax.legend(ncol=3,loc='lower left',fontsize=8,frameon=False)
    export_figure(fig,figs,'S03_simulation_interval_coverage')
    # Figure: identification interval when support is deliberately absent
    d=pd.DataFrame(supportrows);fig,ax=plt.subplots(figsize=(8.3,3.5))
    for i,rr in enumerate(d.itertuples()):
        ax.plot([rr.sharp_lower,rr.sharp_upper],[i,i],lw=6,color=COLORS['design'])
        ax.plot([rr.sharp_lower,rr.sharp_upper],[i,i],ls='',marker='|',ms=13,color='#222')
        ax.scatter(rr.truth,i,color=COLORS['adjusted'],marker='D',s=35,zorder=4)
    ax.set_yticks(range(len(d)),d.scenario.str.replace('_',' ',regex=False))
    ax.set_xlabel('Population fairness gap (sharp identified interval)')
    ax.axvline(0,color='#777',lw=.8,ls=':')
    ax.set_title('Support failure widens identification, irrespective of sample size')
    export_figure(fig,figs,'S04_support_and_partial_identification')
    # Monitor overconfident fairness certifications in repeated sampling.
    fig,ax=plt.subplots(figsize=(8.3,4.4))
    decisions=('pseudo','hier','adjusted')
    for j,m in enumerate(decisions):
        rows=summary[(summary.method==m)&(summary.scenario.isin(sc))]
        yvalues=[float(rows[rows.scenario==s].false_low_risk_rate.iloc[0]) for s in sc]
        ax.plot(range(len(sc)),yvalues,marker=['o','s','D'][j],ms=5,
                color=COLORS[m],label=m)
    ax.axhline(.10,color='#666',lw=1,ls=':',label='Illustrative risk threshold 0.10')
    ax.set_xticks(range(len(sc)),[s.replace('_',' ').title() for s in sc])
    ax.set_ylim(-.02,1.02);ax.set_ylabel('Fraction falsely certified low risk')
    ax.set_title('Safety audit: false low-risk declarations (conditional on scenario)')
    ax.grid(axis='y');ax.legend(frameon=False)
    export_figure(fig,figs,'S05_false_fairness_certifications')
    return summary


def acs_features_and_labels(raw):
    from folktables import ACSEmployment
    x,y,g=ACSEmployment.df_to_numpy(raw)
    y=np.asarray(y,dtype=np.int8);g=np.asarray(g,dtype=np.int16)
    # ACSEmployment uses identity preprocess. Verify official definition and row order.
    if len(raw)!=len(y) or len(raw)!=len(x):
        raise AssertionError('Folktables row count changed; audit weights may be misaligned')
    if not np.array_equal(y,(raw['ESR'].to_numpy()==1).astype(np.int8)):
        raise AssertionError('Folktables labels are not aligned to raw ESR')
    if not np.array_equal(g,raw['RAC1P'].to_numpy().astype(np.int16)):
        raise AssertionError('Folktables protected group differs from raw RAC1P')
    return np.nan_to_num(np.asarray(x,dtype=float),nan=-1,posinf=-1,neginf=-1),y,g


def age_sex_cells(raw):
    age=pd.to_numeric(raw['AGEP'],errors='coerce').to_numpy(dtype=float)
    sex=pd.to_numeric(raw['SEX'],errors='coerce').to_numpy(dtype=float)
    if not np.all(np.isfinite(age)) or not np.all(np.isin(sex,[1,2])):
        raise ValueError('Unexpected ACS age or SEX coding; do not silently recode')
    return (np.where(age<35,0,np.where(age<55,1,2))*2+(sex==2)).astype(np.int8)


# The three states are deliberately pre-registered in the analysis.
STATE_FIPS = {'AL': '01', 'CA': '06', 'TX': '48'}


def acs_csv_path(year, state, cache, attempts=4):
    """Return validated, cached *official* ACS 1-year PUMS person CSV.

    Folktables' current helper catches its HTTP/ZIP exception and may then
    return an absent path. This downloader checks both the HTTP response and
    the extracted CSV, retries recoverable errors, and never trusts a bad file.
    The same official Census ZIP and CSV names are used by Folktables.
    """
    import requests
    if state not in STATE_FIPS:
        raise ValueError(f'Unsupported state {state}: allowed {list(STATE_FIPS)}')
    folder=Path(cache)/str(year)/'1-Year'; folder.mkdir(parents=True,exist_ok=True)
    filename=f'psam_p{STATE_FIPS[state]}.csv'
    final=folder/filename
    essentials={'ESR','PWGTP','RAC1P','AGEP','SEX'}
    from folktables import ACSEmployment
    essentials.update(ACSEmployment.features)
    def verify(path):
        if not path.is_file() or path.stat().st_size<5000: return False
        try:
            cols=set(pd.read_csv(path,nrows=0).columns)
            return essentials.issubset(cols)
        except Exception: return False
    if verify(final):
        print(f'[ACS] Using validated cache: {final}',flush=True)
        return final
    if final.exists():
        final.unlink()
    url=f'https://www2.census.gov/programs-surveys/acs/data/pums/{year}/1-Year/csv_p{state.lower()}.zip'
    partial=folder/(filename+'.part')
    errors=[]
    for attempt in range(1,attempts+1):
        try:
            print(f'[ACS] Download attempt {attempt}/{attempts}: {url}',flush=True)
            with requests.get(url,stream=True,timeout=(30,180),headers={'User-Agent':'BPAI-Academic-Research/1.1'}) as r:
                r.raise_for_status()
                with open(partial,'wb') as handle:
                    for chunk in r.iter_content(chunk_size=2**20):
                        if chunk:handle.write(chunk)
            if not zipfile.is_zipfile(partial):
                raise IOError('Census response is not a valid ZIP archive')
            with zipfile.ZipFile(partial) as zf:
                names=[x for x in zf.namelist() if x.lower().endswith('/'+filename.lower()) or x.lower()==filename.lower()]
                if len(names)!=1:
                    raise IOError(f'Missing exact {filename} in downloaded ZIP: {zf.namelist()[:12]}')
                tmp=folder/(filename+'.extracting')
                with zf.open(names[0]) as src, open(tmp,'wb') as dst:
                    shutil.copyfileobj(src,dst,length=2**20)
                if not verify(tmp):
                    raise IOError('Extracted Census CSV lacks required Folktables fields')
                tmp.replace(final)
            partial.unlink(missing_ok=True)
            return final
        except (requests.RequestException, OSError, zipfile.BadZipFile, ValueError) as exc:
            errors.append(f'Attempt {attempt}: {type(exc).__name__}: {exc}')
            partial.unlink(missing_ok=True)
            (folder/(filename+'.extracting')).unlink(missing_ok=True)
            if attempt<attempts: time.sleep(min(3*attempt,9))
    raise RuntimeError('Official Census download failed; no synthetic replacement was used.\n'
                       + '\n'.join(errors)
                       + f'\nDownload URL: {url}')


def iter_acs_chunks(year,state,cache,audit=False,chunksize=50000):
    """Read exactly the Folktables features plus necessary survey fields.
    Never materialize California PUMS or all replicate weights in one table.
    """
    from folktables import ACSEmployment
    feat=list(ACSEmployment.features)
    needed=list(dict.fromkeys(feat+['ESR','RAC1P','AGEP','SEX','PWGTP']))
    if audit: needed += [f'PWGTP{r}' for r in range(1,81)]
    dtype={c:'float32' for c in feat+['ESR','RAC1P','AGEP','SEX']}
    dtype.update({c:'float64' for c in needed if c.startswith('PWGTP')})
    path=acs_csv_path(year,state,cache)
    for chunk in pd.read_csv(path,usecols=needed,dtype=dtype,
                             chunksize=chunksize,low_memory=False,
                             na_values=[' ',''],keep_default_na=True):
        if len(chunk):yield chunk


def train_fixed_classifier(cache,year=2017,state='AL',seed=2088):
    """Frozen classifier trained on official 2017 AL ACSEmployment microdata."""
    from folktables import ACSEmployment
    print(f'[ACS] Loading training wave {year}: {state}',flush=True)
    features=[]; labels=[]; sample_weights=[]
    for raw in iter_acs_chunks(year,state,cache,audit=False):
        x,y,_=acs_features_and_labels(raw)
        features.append(x);labels.append(y)
        sample_weights.append(raw['PWGTP'].to_numpy(float))
    if not features: raise RuntimeError('ACS training data returned no records')
    X=np.concatenate(features);y=np.concatenate(labels)
    weights=np.concatenate(sample_weights)
    if np.any(~np.isfinite(weights)) or np.any(weights<=0):
        raise ValueError('Invalid ACS training survey weights')
    weights=weights/weights.mean()
    names=list(ACSEmployment.features)
    cols=[i for i,n in enumerate(names) if n not in ('RAC1P','SEX')]
    age=names.index('AGEP')
    categorical=[i for i in cols if i!=age]
    pre=ColumnTransformer([
       ('continuous',StandardScaler(),[age]),
       ('categorical',OneHotEncoder(handle_unknown='ignore'),categorical)],
        remainder='drop',sparse_threshold=0.3)
    clf=Pipeline([('preprocess',pre),('classifier',LogisticRegression(
            C=1.0,solver='lbfgs',max_iter=700,random_state=seed))])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        clf.fit(X,y,classifier__sample_weight=weights)
    for item in caught:
        if issubclass(item.category,Warning):
            print(f'[ACS training warning] {item.category.__name__}: {item.message}',flush=True)
    card={'train_state':state,'train_year':year,'n_train':int(len(y)),
          'prevalence_sample':float(y.mean()),
          'prevalence_weighted':float(np.average(y,weights=weights)),
          'features_used':[names[i] for i in cols],
          'features_excluded':['RAC1P','SEX'],
          'classification_threshold':0.5,'trained_on_train_wave_only':True}
    print(f'[ACS] Frozen classifier trained on {len(y):,} records',flush=True)
    del X,y,features,labels,sample_weights,weights;gc.collect()
    return clf,card


def audit_acs_state(model,year,state,cache,output,ndraw=2500,
                    eps=.05,seed=2088,min_pos=10):
    """Chunk-wise exact Hájek, 80 SDR replicates and cell totals.

    The frozen classifier is evaluated on every released person in the
    specified state-year. For efficiency, full records never enter memory.
    """
    print(f'[ACS] Auditing {year} {state} with streaming 80-replicate SDR',flush=True)
    from folktables import ACSEmployment
    repcols=[f'PWGTP{i}' for i in range(1,81)]
    ntot=0; total_weight=0.;total_wy=0.;total_wcorrect=0.;total_wbrier=0.
    n_group=np.zeros(2,int);npos=np.zeros(2,int)
    # Within positive outcome and group, exact sample/weighted success totals.
    uwden=np.zeros(2);uwnum=np.zeros(2);wd=np.zeros(2);wn=np.zeros(2)
    rd=np.zeros((2,80));rn=np.zeros((2,80))
    # 6 age-sex cells within each race; 12 in total.
    cnt=np.zeros((2,NC),dtype=np.int64)
    wden=np.zeros((2,NC));wnum=np.zeros((2,NC))
    thr=np.array([.3,.4,.5,.6,.7])
    threshold_numerator=np.zeros((2,len(thr)))
    auc_y=[];auc_score=[];auc_weight=[]
    for batch,raw in enumerate(iter_acs_chunks(year,state,cache,audit=True)):
        X,y,g=acs_features_and_labels(raw)
        c=age_sex_cells(raw)
        n=len(y);ntot+=n
        w=raw['PWGTP'].to_numpy(float)
        repl=raw[repcols].to_numpy(dtype=np.float64,copy=False)
        if np.any(~np.isfinite(w)) or np.any(w<=0):
            raise ValueError(f'Invalid main ACS weights in {state} chunk {batch}')
        if not np.all(np.isfinite(repl)) or np.any(repl<0):
            raise ValueError(f'Invalid SDR replicate weights in {state} chunk {batch}')
        score=model.predict_proba(X)[:,1]
        z=(score>=.5)
        total_weight+=float(w.sum());total_wy+=float(np.dot(w,y))
        total_wcorrect+=float(np.dot(w,z==y))
        total_wbrier+=float(np.dot(w,(score-y)**2))
        auc_y.append(y);auc_score.append(score.astype(np.float32))
        auc_weight.append(w)
        for j,code in enumerate((1,2)):
            k_group=(g==code)
            k=k_group&(y==1)
            n_group[j]+=int(k_group.sum());npos[j]+=int(k.sum())
            uwden[j]+=float(k.sum());uwnum[j]+=float(z[k].sum())
            wd[j]+=float(w[k].sum());wn[j]+=float(np.dot(w[k],z[k]))
            r=repl[k,:]
            rd[j]+=r.sum(axis=0)
            rn[j]+=r[z[k],:].sum(axis=0)
            cnt[j]+=np.bincount(c[k],minlength=NC)
            wden[j]+=np.bincount(c[k],weights=w[k],minlength=NC)
            wnum[j]+=np.bincount(c[k],weights=w[k]*z[k],minlength=NC)
            threshold_numerator[j]+=np.array([w[k&(score>=t)].sum() for t in thr])
        if batch%3==0:
            print(f'[ACS {state}] processed {ntot:,} rows',flush=True)
        del raw,X,y,g,c,w,repl,score,z
    if ntot==0 or np.any(wd<=0) or np.any(rd<=0):
        raise ValueError(f'{state} has empty positive-outcome group/replicate denominator')
    rates=wn/wd;unrates=uwnum/uwden
    d_naive=float(unrates[0]-unrates[1]);d_weight=float(rates[0]-rates[1])
    rr=rn/rd;replicate_gaps=rr[0]-rr[1]
    var=sdr_variance(d_weight,replicate_gaps)
    se=float(np.sqrt(var))
    rate_ses=[float(np.sqrt(sdr_variance(rates[j],rr[j]))) for j in range(2)]
    auc_y=np.concatenate(auc_y);auc_score=np.concatenate(auc_score)
    auc_weight=np.concatenate(auc_weight)
    # AUC is computed exactly over all survey persons (not a subsample).
    auc=float(roc_auc_score(auc_y,auc_score,sample_weight=auc_weight))
    del auc_y,auc_score,auc_weight;gc.collect()
    mean_weight=total_weight/ntot
    den=wden/mean_weight;yes=wnum/mean_weight
    qr=safe_ratio(wden,wden.sum(axis=1,keepdims=True))
    theta_point=safe_ratio(wnum,wden)
    support=cnt>=min_pos
    rng=np.random.default_rng(seed)
    q=composition_draws(den,ndraw,rng)
    theta_p=pseudo_draws(den,yes,ndraw,rng)
    theta_h,info=hierarchical_laplace(den,yes,ndraw,rng)
    gap_p=posterior_gap(theta_p,q)
    gap_h=posterior_gap(theta_h,q)
    gap_adj=design_adjust_gap(gap_h,d_weight,var)
    cert=certificate(q,theta_h,eps=eps)
    if not support.all():
        cert['status']='empirical_support_warning_no_certification'
    risks=summarize_draws(gap_h,eps)
    risks_p=summarize_draws(gap_p,eps)
    risks_adj=summarize_draws(gap_adj,eps)
    metric={
        'state':state,'year':year,'n_audit':ntot,
        'n_group_white':int(n_group[0]),'n_group_black':int(n_group[1]),
        'positive_white':int(npos[0]),'positive_black':int(npos[1]),
        'population_weight_total':float(total_weight),
        'weighted_y_prevalence':float(total_wy/total_weight),
        'weighted_accuracy':float(total_wcorrect/total_weight),
        'weighted_brier':float(total_wbrier/total_weight),
        'weighted_auc':auc,
        'gap_unweighted':d_naive,'gap_weighted':d_weight,
        'gap_sdr_se':se,'gap_sdr_lower95':d_weight-1.96*se,
        'gap_sdr_upper95':d_weight+1.96*se,
        'hier_posterior_mean':risks['mean'],'hier_lower95':risks['lo'],
        'hier_upper95':risks['hi'],'hier_risk_epsilon':risks['risk'],
        'pseudo_mean':risks_p['mean'],'pseudo_lower95':risks_p['lo'],
        'pseudo_upper95':risks_p['hi'],'pseudo_risk_epsilon':risks_p['risk'],
        'aligned_mean':risks_adj['mean'],'aligned_lower95':risks_adj['lo'],
        'aligned_upper95':risks_adj['hi'],'aligned_risk_epsilon':risks_adj['risk'],
        'certificate_status':cert['status'],'certificate_B':cert['B'],
        'certificate_joint_empirical_probability':cert['joint_prob'],
        'certificate_reference_gap':cert['refgap'],
        'all_cells_at_least_min_positive':bool(support.all()),
        'min_cell_positive_count':int(cnt.min()),
        'laplace_optimizer_success':info['converged'],
        'hier_to_SDR_variance_ratio':float(risks['sd']**2/var) if var>0 else np.nan,
        'illustrative_review_cost_over_harm':0.10,
        'working_audit_action':('approve_for_further_review' if risks['risk']<=.10 and support.all()
                                 else 'defer_for_review')}
    group_rows=[];cell_rows=[]
    for j,code in enumerate((1,2)):
        group_rows.append({'state':state,'group':G_NAMES[j],
            'TPR_unweighted':float(unrates[j]),'TPR_weighted':float(rates[j]),
            'TPR_SDR_SE':float(rate_ses[j]),'weighted_population_positive':float(wd[j]),
            'positive_sample_n':int(npos[j])})
        for k,label in enumerate(C_NAMES):
            cell_rows.append({'state':state,'race':G_NAMES[j],'cell':label,
                'positive_n':int(cnt[j,k]),
                'positive_weighted_total':float(wden[j,k]),
                'q_positive_weighted':float(qr[j,k]),
                'theta_positive_weighted':float(theta_point[j,k]),
                'supported_min_n':bool(support[j,k])})
    sensitivity=[]
    for prior_scale in (.4,.8,1.6):
        rrng=np.random.default_rng(seed+int(prior_scale*1000))
        tt,diagnostic=hierarchical_laplace(den,yes,ndraw,rrng,prior_scale)
        ss=summarize_draws(posterior_gap(tt,q),eps)
        sensitivity.append({'state':state,'prior_scale':prior_scale,
            'posterior_mean':ss['mean'],'lower95':ss['lo'],
            'upper95':ss['hi'],'violation_probability':ss['risk']})
    thresh_rows=[]
    for j,t in enumerate(thr):
        dd=float(threshold_numerator[0,j]/wd[0]-threshold_numerator[1,j]/wd[1])
        thresh_rows.append({'state':state,'threshold':float(t),'weighted_gap':dd})
    low,high,unknown=ignorance_bounds(qr,np.nan_to_num(theta_point,nan=0),support)
    metric.update({'ignorance_lower_conditional':low,
                   'ignorance_upper_conditional':high,
                   'unsupported_q_mass':unknown})
    out=Path(output);(out/'posterior_draws').mkdir(parents=True,exist_ok=True)
    np.savez_compressed(out/'posterior_draws'/f'{state}_posterior_gaps.npz',
                        hierarchical=gap_h,pseudo=gap_p,sdraligned=gap_adj,
                        q_posterior=q,theta_hierarchical=theta_h,
                        design_replicate_gaps=replicate_gaps)
    data={'q':qr,'theta':theta_point,'support':support,'state':state}
    print(f'[ACS] {state}: N={ntot:,}, weighted gap={d_weight:+.4f} '
          f'(SDR SE={se:.4f}); posterior rho={risks["risk"]:.3f}',flush=True)
    return metric,group_rows,cell_rows,sensitivity,thresh_rows,data


def audit_geographic_transport(aggregates,figdir,tabdir):
    source=aggregates.get('AL')
    if source is None:
        raise RuntimeError('Need AL as specified reference region')
    rows=[]
    for st,d in aggregates.items():
        if st=='AL': continue
        # No fabricated smoothing of unknown cells; positive-support required.
        valid=np.isfinite(source['theta'])&np.isfinite(d['theta'])
        if not np.all(valid):
            rows.append({'source':'AL','target':st,'identified':False,
                         'reason':'At least one positive-outcome cell without observed support'})
            continue
        r=decomposition(source['q'],source['theta'],d['q'],d['theta'])
        rows.append({'source':'AL','target':st,'identified':True,**r})
    frame=pd.DataFrame(rows)
    table(frame,tabdir,'ACS_transport_decomposition')
    complete=frame[frame['identified']==True]
    if not complete.empty:
        fig,ax=plt.subplots(figsize=(7.7,4.1))
        ix=np.arange(len(complete)); labels=complete['target'].to_list()
        for k,(name,c) in enumerate([('composition_gap',COLORS['design']),
                                      ('rate_gap',COLORS['hier'])]):
            v=complete[name].to_numpy(float)
            ax.bar(ix+(k-.5)*.36,v,width=.32,label=name.replace('_',' ').title(),color=c)
        ax.plot(ix,complete['change'],marker='D',lw=0,ms=5,color='black',label='Exact total change')
        ax.axhline(0,color='#454545',lw=.9)
        ax.set_xticks(ix,[f'AL → {x}' for x in labels]);ax.set_ylabel('Change in White minus Black TPR')
        ax.set_title('Source-to-target gap: composition versus within-cell change')
        ax.legend(frameon=False,fontsize=9);ax.grid(axis='y')
        export_figure(fig,figdir,'R04_ACS_transport_decomposition')
    return frame


def plot_acs(metrics,groups,sensitivity,thresholds,out,eps):
    metrics=metrics.sort_values('state');figs=Path(out)/'figures'
    order=[s for s in ('AL','CA','TX') if s in set(metrics.state)]
    m=metrics.set_index('state').loc[order].reset_index()
    x=np.arange(len(order));
    fig,ax=plt.subplots(figsize=(8.1,4.4))
    for col,shift,title,c in [('gap_unweighted',-.17,'Unweighted sample',COLORS['naive']),
                               ('gap_weighted',+.17,'ACS-weighted population',COLORS['design'])]:
        v=m[col].to_numpy(float)
        ax.bar(x+shift,v,width=.29,label=title,color=c)
    se=m['gap_sdr_se'].to_numpy(float)
    ax.errorbar(x+.17,m['gap_weighted'],yerr=1.96*se,fmt='none',
                ecolor='#142b44',capsize=4,lw=1.2,label='SDR 95% normal interval')
    ax.axhline(0,lw=.8,color='#555');ax.axhspan(-eps,eps,color='#777',alpha=.06,zorder=0)
    ax.set_xticks(x,order);ax.set_ylabel('White minus Black true-positive rate')
    ax.set_title('Sample fairness versus the survey-weighted population target')
    ax.legend(frameon=False,loc='best',fontsize=8.5);ax.grid(axis='y')
    export_figure(fig,figs,'R01_ACS_weighted_vs_naive')

    gd=groups.copy();fig,ax=plt.subplots(figsize=(8.1,4.4))
    for j,grp in enumerate(G_NAMES):
        vals=gd[gd['group']==grp].set_index('state').loc[order]
        yy=vals.TPR_weighted.to_numpy(float)
        vv=vals.TPR_SDR_SE.to_numpy(float)
        ax.errorbar(x+(.045 if j else -.045),yy,yerr=1.96*vv,
                     marker='o' if j==0 else 's',ms=6,capthick=1,capsize=3,
                     color=COLORS['design'] if j==0 else COLORS['adjusted'],label=grp)
    ax.set_xticks(x,order);ax.set_ylabel('Weighted true-positive rate')
    ax.set_title('Target-population TPR by race and geographic domain')
    ax.set_ylim(0,1);ax.grid(axis='y');ax.legend(frameon=False,fontsize=9)
    export_figure(fig,figs,'R02_ACS_group_TPRs')

    fig,ax=plt.subplots(figsize=(8.4,4.5))
    for j,r in enumerate(m.itertuples()):
        yy=j
        ax.plot([r.gap_sdr_lower95,r.gap_sdr_upper95],[yy+.17,yy+.17],
                color=COLORS['design'],lw=3,solid_capstyle='butt')
        ax.plot(r.gap_weighted,yy+.17,'o',color=COLORS['design'],ms=6)
        ax.plot([r.hier_lower95,r.hier_upper95],[yy-.17,yy-.17],
                color=COLORS['hier'],lw=3,solid_capstyle='butt')
        ax.plot(r.hier_posterior_mean,yy-.17,'s',color=COLORS['hier'],ms=5)
    ax.axvline(0,color='#555',lw=.7)
    ax.set_yticks(np.arange(len(order)),order);ax.set_ylabel('2018 audit state')
    ax.set_xlabel('White minus Black TPR gap')
    ax.set_title('Design-SDR intervals and hierarchical pseudo-posterior intervals')
    ax.text(.02,-.26,'Top line: SDR  |  Bottom line: hierarchical Laplace',
            transform=ax.transAxes,fontsize=9,color='#515d68')
    ax.grid(axis='x');export_figure(fig,figs,'R03_ACS_uncertainty_comparison')

    fig,ax=plt.subplots(figsize=(8.2,4.5))
    xs=np.linspace(0,.20,81)
    for st in order:
        a=np.load(Path(out)/'posterior_draws'/f'{st}_posterior_gaps.npz')
        d=a['hierarchical'];probs=np.array([np.mean(abs(d)>j) for j in xs])
        ax.plot(xs,probs,color=COLORS[st],label=st)
    ax.axvline(eps,color='#333',ls='--',lw=1,label=f'Predefined ε={eps:.2f}')
    ax.axhline(.1,color='#777',ls=':',lw=1,label='Illustrative review-cost ratio 0.10')
    ax.set(xlabel='Fairness violation tolerance ε',ylabel='Working posterior P(|Δ| > ε)',ylim=(0,1))
    ax.set_title('Decision uncertainty changes with the substantive fairness tolerance')
    ax.grid(alpha=.16);ax.legend(frameon=False)
    export_figure(fig,figs,'R05_ACS_fairness_violation_curves')

    s=sensitivity.copy()
    fig,ax=plt.subplots(figsize=(8.3,4.4))
    for st in order:
        a=s[s.state==st].sort_values('prior_scale')
        ax.errorbar(a.prior_scale,a.posterior_mean,
             yerr=np.vstack([a.posterior_mean-a.lower95,a.upper95-a.posterior_mean]),
             marker='o',ms=5,capsize=3,color=COLORS[st],label=st)
    ax.axhline(0,color='#666',lw=.8)
    ax.set_xscale('log',base=2);ax.set_xticks([.4,.8,1.6],['0.4','0.8','1.6'])
    ax.set_xlabel('Gaussian prior effect scale')
    ax.set_ylabel('Hierarchical pseudo-posterior gap and 95% interval')
    ax.set_title('Sensitivity to partial-pooling prior strength')
    ax.grid(axis='y');ax.legend(frameon=False)
    export_figure(fig,figs,'R06_ACS_prior_sensitivity')

    th=thresholds.copy()
    fig,ax=plt.subplots(figsize=(7.9,4.1))
    for st in order:
        a=th[th.state==st].sort_values('threshold')
        ax.plot(a.threshold,a.weighted_gap,marker='o',color=COLORS[st],label=st,ms=5)
    ax.axvline(.5,color='#444',ls=':',lw=1)
    ax.axhline(0,color='#777',lw=.8)
    ax.set_xlabel('Frozen model decision threshold')
    ax.set_ylabel('Survey-weighted true-positive-rate gap')
    ax.set_title('Sensitivity to the classification decision threshold')
    ax.grid(alpha=.17);ax.legend(frameon=False)
    export_figure(fig,figs,'R07_ACS_threshold_sensitivity')


def run_acs(out,cache,year=2018,train_year=2017,states=('AL','CA','TX'),
            ndraw=2500,eps=.05,seed=2088,min_pos=10):
    if train_year>=year:
        raise ValueError('A prior-year training wave is required to avoid train/audit leakage')
    if 'AL' not in states: raise ValueError('Geographic decomposition uses AL as reference')
    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)
    out=Path(out);(out/'tables').mkdir(parents=True,exist_ok=True)
    model,modelcard=train_fixed_classifier(cache,train_year,'AL',seed)
    (out/'classifier_design.json').write_text(json.dumps(modelcard,indent=2))
    metrics=[];groups=[];cells=[];sens=[];threshold=[];aggregates={}
    for i,st in enumerate(states):
        metric,gr,cr,sn,tt,data=audit_acs_state(model,year,st,cache,out,ndraw,
                                                   eps,seed+i*202,min_pos)
        metrics.append(metric);groups.extend(gr);cells.extend(cr)
        sens.extend(sn);threshold.extend(tt);aggregates[st]=data
        table(pd.DataFrame(metrics),out/'tables','ACS_state_summary')
    mt=pd.DataFrame(metrics);gr=pd.DataFrame(groups)
    ss=pd.DataFrame(sens);tt=pd.DataFrame(threshold)
    table(mt,out/'tables','ACS_state_summary')
    table(gr,out/'tables','ACS_group_TPRs')
    table(pd.DataFrame(cells),out/'tables','ACS_group_by_cell')
    table(ss,out/'tables','ACS_prior_sensitivity')
    table(tt,out/'tables','ACS_threshold_sensitivity')
    transport=audit_geographic_transport(aggregates,out/'figures',out/'tables')
    plot_acs(mt,gr,ss,tt,out,eps)
    # A numerical narrative is generated ONLY from actual output rows.
    lines=['# ACS 2018 results: automatically generated descriptive summary', '',
           'Do not interpret the predictive gap as causal employment discrimination.',
           'Intervals labelled SDR use ACS replicate weights; hierarchical intervals are',
           'working weighted pseudo-posterior Laplace approximations.', '']
    for rec in mt.itertuples():
        lines.extend([f'## {rec.state} (2018; {rec.n_audit:,} PUMS person records)',
            f'- Naive sample fairness gap: {rec.gap_unweighted:+.4f}.',
            f'- Survey-weighted gap: {rec.gap_weighted:+.4f}; SDR SE {rec.gap_sdr_se:.4f}; '
              f'95% Wald interval [{rec.gap_sdr_lower95:+.4f}, {rec.gap_sdr_upper95:+.4f}].',
            f'- Hierarchical weighted pseudo-posterior gap: mean {rec.hier_posterior_mean:+.4f}; '
              f'95% model-conditional interval [{rec.hier_lower95:+.4f}, {rec.hier_upper95:+.4f}].',
            f'- Working posterior P(|gap|>{eps:.3f}) = {rec.hier_risk_epsilon:.3f}; '
              f'joint-envelope certificate status: {rec.certificate_status}.',
            f'- Minimum number of observed positives in group-by-cell table: '
              f'{rec.min_cell_positive_count}.', ''])
    lines.extend(['## Interpretation safeguards','',
       '- These figures are observational summaries of reported employment status.',
       '- Posterior values are conditional on the stated likelihood/prior and are not guaranteed design-calibrated.',
       '- The trained model is fixed; uncertainty in training is not propagated.',
       '- Cross-region decomposition is descriptive, not causal.',
       '- Edit all interpretive claims after visually inspecting the diagnostic figures and prior-sensitivity tables.'])
    (out/'ACS_results_narrative.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
    # Machine-readable results WITHOUT uploading individual microdata to the ZIP.
    return mt


def unit_tests():
    # Mathematical unit tests, distinct from external-data empirical validation.
    g=np.array([1,1,2,2]);y=np.ones(4,dtype=int);z=np.array([1,0,1,0]);w=np.array([2.,1.,1.,3.])
    gap,rates,_=weighted_gap(y,z,g,w)
    assert np.isclose(gap,2/3-1/4)
    fake=np.full(80,gap+.02)
    assert np.isclose(sdr_variance(gap,fake),4*.02**2)
    t=np.array([[.9,.5],[.85,.55]]);p=np.array([[.2,.8],[.8,.2]]);q=np.array([[.7,.3],[.3,.7]])
    obj=decomposition(p,t,q,t)
    assert np.isclose(obj['delta_source'],-.21)
    assert np.isclose(obj['delta_target'],+.14)
    assert np.isclose(obj['rate_gap'],0)
    lo,hi,mass=ignorance_bounds(q,t,np.array([[1,0],[1,1]],dtype=bool))
    assert hi>=obj['delta_target']>=lo
    rg=np.random.default_rng(13)
    qq=np.stack([rg.dirichlet([11,10],size=450),rg.dirichlet([10,11],size=450)],axis=1)
    tt=rg.beta(5,5,size=(450,2,2))
    cb=certificate(qq,tt)
    assert cb['joint_prob']>=.94
    d=np.ones((2,NC))*90;s=np.ones((2,NC))*45
    hh,info=hierarchical_laplace(d,s,80,rg)
    assert hh.shape==(80,2,NC) and np.all((hh>0)&(hh<1))
    print('[unit tests] All mathematical and computational tests passed.',flush=True)


def write_manifest(out,args):
    packages={}
    for p in ['numpy','pandas','scipy','scikit-learn','matplotlib','folktables']:
        try: packages[p]=version(p)
        except PackageNotFoundError: packages[p]='not installed'
    files=[]
    root=Path(out)
    for p in root.rglob('*'):
        if p.is_file() and p.name!='manifest.json':
            files.append({'path':str(p.relative_to(root)),
                          'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),
                          'bytes':p.stat().st_size})
    manifest={'created_at_utc':datetime.now(timezone.utc).isoformat(),
              'python':platform.python_version(),'command':vars(args),
              'seed':args.seed,'software_versions':packages,
              'files':files,
              'scientific_cautions':[
                  'Weighted pseudo-posteriors are not exact design-calibrated Bayesian posteriors.',
                  'Laplace approximation has fixed prior scales and conditional-on-model uncertainty.',
                  'SDR variance is a design-based anchor for a frozen classifier, not posterior draws.',
                  'ACSEmployment concerns recorded employment status, not causal hiring discrimination.',
                  'ACS audit is full-state 2018 survey records, no random test split.',
                  'Numerical coverage in simulation is conditional on a synthetic finite population and Poisson sampling.',
                  'Cell support warnings are empirical, not proof of structural positivity failure.'
              ]}
    (root/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')


def zip_results(out):
    root=Path(out); destination=root.parent/(root.name+'_download.zip')
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p in sorted(root.rglob('*')):
            if p.is_file():z.write(p,arcname=str(Path(root.name)/p.relative_to(root)))
    return destination


def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--mode',choices=['smoke','simulation','acs','all'],default='all')
    parser.add_argument('--out',default='BPAI_results')
    parser.add_argument('--cache',default='BPAI_ACS_cache')
    parser.add_argument('--seed',type=int,default=2088)
    parser.add_argument('--reps',type=int,default=120)
    parser.add_argument('--population',type=int,default=24000)
    parser.add_argument('--draws',type=int,default=1500)
    parser.add_argument('--acs-draws',type=int,default=3000)
    parser.add_argument('--epsilon',type=float,default=.05)
    parser.add_argument('--year',type=int,default=2018)
    parser.add_argument('--train-year',type=int,default=2017)
    parser.add_argument('--states',nargs='+',default=['AL','CA','TX'])
    parser.add_argument('--min-cell-positive-n',type=int,default=10)
    parser.add_argument('--no-zip',action='store_true')
    a=parser.parse_args(argv)
    if not (0<a.epsilon<1) or a.reps<2 or a.draws<100 or a.acs_draws<100:
        raise ValueError('Invalid epsilon, number of replications or draws')
    random.seed(a.seed);np.random.seed(a.seed);style()
    unit_tests()
    out=Path(a.out).resolve();out.mkdir(parents=True,exist_ok=True)
    if a.mode in ('smoke','simulation','all'):
        reps=3 if a.mode=='smoke' else a.reps
        pop=5000 if a.mode=='smoke' else a.population
        dr=120 if a.mode=='smoke' else a.draws
        simulate(out/'simulation',reps,pop,dr,a.seed,a.epsilon)
    if a.mode in ('acs','all'):
        run_acs(out/'acs',a.cache,a.year,a.train_year,a.states,
                a.acs_draws,a.epsilon,a.seed,a.min_cell_positive_n)
    write_manifest(out,a)
    if not a.no_zip:
        z=zip_results(out);print(f'OUTPUT_ZIP={z}',flush=True)
    print('COMPLETE',flush=True)


if __name__=='__main__':
    main()


Writing bpai_study.py


In [3]:
# Check the source is physically present and that mathematical tests pass.
import importlib.util, sys
from pathlib import Path
SCRIPT = Path.cwd() / "bpai_study.py"
assert SCRIPT.is_file(), f"Missing source code: {SCRIPT}; run the previous cell"
if str(Path.cwd()) not in sys.path: sys.path.insert(0, str(Path.cwd()))
spec = importlib.util.spec_from_file_location("bpai_study_v2", SCRIPT)
b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(b)
b.style()
b.unit_tests()
print(f"SOURCE_READY: {SCRIPT} ({SCRIPT.stat().st_size:,} bytes)")

[unit tests] All mathematical and computational tests passed.
SOURCE_READY: /content/bpai_study.py (55,134 bytes)


## Run the experiment

The runner calls Python functions directly, **not** `subprocess.run(check=True)`, so the actual Python exception is printed. Both research stages are separate. A failing ACS download leaves the successful simulation stage intact. Even on error, the output ZIP contains `STAGE_STATUS.json` and error tracebacks.

- `pilot`: 20 repeats × 12,000 finite population, plus full real ACS processing
- `full`: 250 repeats × 30,000 finite population, plus full real ACS processing
- `acs_only`: full ACS; no simulations
- `simulation_only`: 250 simulation repetitions; no ACS download

The `pilot` setting changes **simulation replication count and posterior Monte Carlo draws**, not the size of the real ACS evaluation dataset: the latter is always the full released state files.

In [5]:
import json, traceback, datetime, sys, os
from pathlib import Path

RUN_PROFILE = "full"   # "full", "pilot", "acs_only", "simulation_only"
assert RUN_PROFILE in ("full", "pilot", "acs_only", "simulation_only")
BASE = Path.cwd()
OUT = BASE / f"BPAI_results_{RUN_PROFILE}_v2"
CACHE = BASE / "BPAI_ACS_cache"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "logs").mkdir(parents=True, exist_ok=True)

COMMON = ['--out', str(OUT), '--cache', str(CACHE), '--seed', '2088',
          '--year','2018','--train-year','2017','--states','AL','CA','TX',
          '--epsilon','0.05','--no-zip']
SIM_FULL = ['--reps','250','--population','30000','--draws','1300']
SIM_PILOT = ['--reps','20','--population','12000','--draws','400']
ACS_FULL = ['--acs-draws','3500']
ACS_PILOT = ['--acs-draws','800']
STAGES = []
if RUN_PROFILE in ('pilot','full','simulation_only'):
    STAGES.append(('simulation', SIM_PILOT if RUN_PROFILE=='pilot' else SIM_FULL))
if RUN_PROFILE in ('pilot','full','acs_only'):
    STAGES.append(('acs', ACS_PILOT if RUN_PROFILE=='pilot' else ACS_FULL))

statuses=[]
for stage, extra in STAGES:
    print('\n'+'='*72, flush=True)
    print(f'RUNNING {stage.upper()}: profile={RUN_PROFILE}', flush=True)
    argv=COMMON+['--mode',stage]+extra
    try:
        b.main(argv)
        statuses.append({'stage':stage,'status':'success','command_args':argv})
        print(f'{stage.upper()} STAGE COMPLETE',flush=True)
    except Exception:
        trace=traceback.format_exc()
        print(f'\n{stage.upper()} FAILED; actual exception follows:',file=sys.stderr,flush=True)
        print(trace,file=sys.stderr,flush=True)
        (OUT/'logs'/f'{stage}_error_traceback.txt').write_text(trace,encoding='utf8')
        statuses.append({'stage':stage,'status':'failed','command_args':argv,
                         'traceback_file':f'logs/{stage}_error_traceback.txt'})
    finally:
        (OUT/'STAGE_STATUS.json').write_text(json.dumps({
          'run_profile':RUN_PROFILE,'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'stages':statuses},indent=2),encoding='utf8')
        archive=b.zip_results(OUT)
        print('Current ZIP:',archive,flush=True)

print('\nFINAL:',[(s['stage'],s['status']) for s in statuses],flush=True)
print('OUTPUT_ZIP:',archive,flush=True)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Not running in Google Colab. The ZIP is available at',archive)

if any(x['status']=='failed' for x in statuses):
    print('IMPORTANT: At least one stage failed. Read the corresponding error '
          'in the ZIP logs folder. Do not describe an incomplete run as a real-data verification.')


RUNNING SIMULATION: profile=full
[unit tests] All mathematical and computational tests passed.
[simulation] near_uniform: 250 draws, population gap = +0.0768
[simulation] informative: 250 draws, population gap = +0.0734
[simulation] strong_informative: 250 draws, population gap = +0.0764
[simulation] prediction_dependent: 250 draws, population gap = +0.0628
[simulation] lack_support: 250 draws, population gap = +0.0615
COMPLETE
SIMULATION STAGE COMPLETE
Current ZIP: /content/BPAI_results_full_v2_download.zip

RUNNING ACS: profile=full
[unit tests] All mathematical and computational tests passed.
[ACS] Loading training wave 2017: AL
[ACS] Using validated cache: /content/BPAI_ACS_cache/2017/1-Year/psam_p01.csv
[ACS training warning] DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
[ACS] Frozen classifier trained on 47,645 records
[ACS] Auditing 2018 AL with streaming 80-replicate SDR
[ACS] Usin


ACS FAILED; actual exception follows:
Traceback (most recent call last):
  File "/tmp/ipykernel_5087/906301883.py", line 31, in <cell line: 0>
    b.main(argv)
    ~~~~~~^^^^^^
  File "/content/bpai_study.py", line 1075, in main
    run_acs(out/'acs',a.cache,a.year,a.train_year,a.states,
    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            a.acs_draws,a.epsilon,a.seed,a.min_cell_positive_n)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/bpai_study.py", line 945, in run_acs
    metric,gr,cr,sn,tt,data=audit_acs_state(model,year,st,cache,out,ndraw,
                            ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                               eps,seed+i*202,min_pos)
                                               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/bpai_study.py", line 686, in audit_acs_state
    raise ValueError(f'Invalid SDR replicate weights in {state} chunk {batch}')
ValueError: Invalid SDR replicate 

Current ZIP: /content/BPAI_results_full_v2_download.zip

FINAL: [('simulation', 'success'), ('acs', 'failed')]
OUTPUT_ZIP: /content/BPAI_results_full_v2_download.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

IMPORTANT: At least one stage failed. Read the corresponding error in the ZIP logs folder. Do not describe an incomplete run as a real-data verification.


## Research interpretation notes

- `simulation/tables/simulation_method_summary.csv` reports Monte Carlo bias, RMSE, interval coverage, coverage MCSE and low-risk misclassification frequency. The `lack_support` scenario tests partial identification, **not** nominal coverage for a point-identified full-population functional.
- `acs/tables/ACS_state_summary.csv` reports unweighted and weighted TPR gaps, official ACS SDR confidence limits, working Bayesian posterior gap intervals, event probabilities and sensitivity flags.
- `acs/tables/ACS_transport_decomposition.csv` separates population composition changes from within-cell changes for AL→CA and AL→TX, describing observed populations, not causal shifts.
- `acs/figures/` and `simulation/figures/` export 300 dpi PNG plus vector PDF/SVG.
- Pseudo-posterior credible intervals and Laplace draws are **model-dependent**; scalar SDR alignment is a sensitivity diagnostic, not an exact Bayesian posterior or a general proof of survey-design validity.
- ACS employment status is observational and does not establish causal discrimination. The classifier is frozen for audit; classifier-training uncertainty is not included.

When using the output for journal publication, examine the result tables, posterior diagnostic fields, simulation error rates, sensitivity analyses and any error logs first. Do not include pilot simulation estimates as final Monte Carlo evidence.